# 📘 Tutorial 1: Gradient Descent as Dynamical System
> This notebook is provided in a clean, non-executed state for readability and reproducibility.

This tutorial marks the beginning of **Part 2**, where we shift focus from gradient construction to **gradient-driven dynamics**.

Here, gradient descent is treated not as a training heuristic, but as a **deterministic dynamical system** governed by geometry, curvature, and step size.

Rather than introducing optimisers, datasets, or neural networks, we study gradient descent in its simplest possible form: repeated application of a local update rule
$$x_{t+1} = x_t - \eta \nabla f(x_t),$$
and ask what this rule does as a function of the objective and the learning rate.

---

**This tutorial is designed to shift perspective**
- from *“gradient descent as an algorithm that minimises loss”*
- to *“gradient descent as a discrete-time dynamical system with stability properties”*.

---

**The emphasis is on developing intuition for**
- how learning rates control contraction, oscillation, and divergence,
- how curvature and conditioning shape trajectories,
- how local linearisation explains global behaviour,
- and why optimisation failures are often predictable from geometry alone.

---

**Key ideas explored include**
- gradient descent as a linear recurrence on quadratic objectives,
- stability conditions and learning-rate bounds,
- eigenvalues as the governing quantities of convergence,
- ill-conditioning and slow directions,
- and how non-convexity introduces multiple basins and unstable fixed points.

---

This tutorial serves as the foundation for
- understanding optimisation dynamics beyond a single step,
- analysing conditioning and preconditioning,
- motivating adaptive methods and momentum,
- and interpreting optimisation behaviour before stochasticity is introduced.

The focus is intentionally not on performance, architectures, or real datasets.

Instead, the goal is to understand **why gradient descent behaves the way it does**—before making it more complicated.

---

**Recommended prerequisites**
- Comfort with tensors and basic gradient concepts (Part 1)
- Familiarity with simple derivatives
- Basic linear algebra (eigenvalues, quadratic forms)

---

**Author**: Angze Li

**Last updated**: 2026-03-05

**Version**: v1.1

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

This cell sets up the environment and defines a few small helper functions used throughout the tutorial.

### dtype
```python
torch.set_default_dtype(torch.float64)
```
We use double-precision `(float64)` by default to improve numerical stability and make gradient dynamics easier to inspect and compare across runs. This is especially helpful when visualising optimisation trajectories.

---

### Controlling Randomness
```python
def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
```
This function fixes the random number generators in both PyTorch and NumPy. By calling `set_seed(0)`, we ensure that:
- random initialisations are reproducible,
- experiments can be rerun and compared reliably,
- observed optimisation behaviour reflects the algorithm, not randomness.

---

### Plot Styling Helper
```python
def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")
```
This is a small utility function to apply consistent styling to plots:
- thicker axes for clarity,
- larger tick labels,
- bold font for readability.
It keeps plotting code clean and ensures a uniform visual style across figures.

### Initialise
```python
set_seed(0)
```
We fix the seed once at the start of the tutorial so that all subsequent experiments are fully reproducible.

In [ ]:
torch.set_default_dtype(torch.float64)

def set_seed(seed=0):
    torch.manual_seed(seed)     #---> Sets PyTorch’s RNG seed
    np.random.seed(seed)        #---> Sets Numpy’s RNG seed

def style_ax(ax):               #---> stylistic helper
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

set_seed(0)

## 1. Gradient Descent Package

In this tutorial, we treat gradient descent as a discrete-time dynamical system rather than a black-box optimiser.
This cell defines a minimal implementation of gradient descent and a helper routine to run it for multiple steps.

---

###  Single Gradient Descent Step
```python
def gd_step(x, grad, lr):
    return x - lr * grad
```
This function implements the core gradient descent update:
$$x_{k+1} = x_k - \eta \nabla L(x_k),$$
where:
- $x_k$ is the current parameter state,
- $\nabla L(x_k)$ is the gradient of the loss at that state,
- $\eta (lr)$ is the learning rate.

Nothing more is happening here — gradient descent is simply repeated application of this rule.

---

### Running Gradient Descent Over Time
```python
@torch.no_grad()
def run_gd(x0, loss_and_grad_fn, lr, steps=50):
```
This function evolves the system forward for a fixed number of steps.
- `x0` is the initial state.
- `loss_and_grad_fn` is a callable that returns both the loss and its gradient:
```python
loss_and_grad_fn(x) → (loss_scalar, gradient_tensor)
```
where
- `lr` is the learning rate.
- `steps` controls how many iterations are performed.
The `@torch.no_grad()` decorator disables autograd tracking inside this loop.

We are manually applying gradients, not differentiating through the optimisation process itself.

---

### What Do We Record?

Inside the loop, we track three quantities:
```python
xs         # parameter trajectory
losses     # loss value at each step
grad_norms # gradient magnitude at each step
```
- `xs` lets us visualise how parameters move in space.
- `losses` show convergence (or divergence).
- `grad_norms` reveal how the optimisation dynamics evolve over time.

Together, these form a complete description of the optimisation trajectory.

In [ ]:
def gd_step(x, grad, lr):
    return x - lr * grad

@torch.no_grad()
def run_gd(x0, loss_and_grad_fn, lr, steps=50):
    x = x0.clone()
    xs, losses, grad_norms = [x.clone()], [], []
    for _ in range(steps):
        L, g = loss_and_grad_fn(x)
        losses.append(float(L))
        grad_norms.append(float(g.norm()))
        x = gd_step(x, g, lr)
        xs.append(x.clone())
    return torch.stack(xs), np.array(losses), np.array(grad_norms)

## 2. Gradient Descent on a 1D Quadratic

We now study gradient descent in its simplest possible setting: a one-dimensional quadratic loss.

### Function studied 

$$L(x) = \tfrac{1}{2} a x^2,\qquad \nabla L(x) = a x.$$

This example is important because:
- it is analytically solvable,
- it exposes the role of the learning rate clearly,
- and it already exhibits **stability vs. divergence**.

---

### Loss and Gradient Definition
```python
def quad1d(a=2.0):
```
The function quad1d returns a callable that computes both:
- the scalar loss `L(x)`,
- and its gradient `\nabla L(x)`.

This mirrors how optimisation works conceptually: the loss defines the geometry, and the gradient defines the local update direction.

---

### Update Rule as a Linear Dynamical System

For this quadratic, one gradient descent step gives:
$$x_{t+1} = x_t - \eta a x_t = (1 - \eta a)\,x_t.$$

This is a linear recurrence, meaning:
- convergence, oscillation, or divergence are determined entirely by the factor $1 - \eta a$.

---

### Learning Rates Tested
```pythonb
lrs = [0.1, 0.9/a, 1.1/a]
```
These choices correspond to:
- small learning rate: stable convergence,
- near the stability boundary: slow or oscillatory behaviour,
- beyond the boundary: divergence.

This makes the effect of $\eta$ visible immediately.

---

### What the Plot Shows

Each curve shows the trajectory $\{x_t\}$ over time for a different learning rate:
- the horizontal axis is the iteration index $t$,
- the vertical axis is the parameter value $x_t$.

Despite the simplicity of the loss, we already see:
- stable decay,
- marginal stability,
- and explosive divergence.

In [ ]:
def quad1d(a=2.0):
    def fn(x):
        # x is scalar tensor
        L = 0.5 * a * x**2
        g = a * x
        return L, g
    return fn

a = 2.0
f = quad1d(a=a)

x0 = torch.tensor(3.0)
lrs = [0.1, 0.9/a, 1.1/a]  # stable, near boundary, unstable
steps = 30

fig, ax = plt.subplots(figsize=(8,6))
for lr in lrs:
    xs, losses, gns = run_gd(x0, f, lr=lr, steps=steps)
    ax.plot(xs.squeeze().numpy(), "-o", linewidth=2.0, label=f"lr={lr:.3f}")
ax.set_title(r"1D quadratic GD trajectory: $x_{t+1}=(1-\eta a)x_t$", fontsize=14, fontweight="bold")
ax.set_xlabel("step $t$", fontsize=14, fontweight="bold")
ax.set_ylabel("$x_t$", fontsize=14, fontweight="bold")
ax.legend(prop={"size": 11, "weight": "bold"})
style_ax(ax)
plt.show()

### Interpreting each curve

#### 🔵 lr = 0.1 (stable, slow)
- Here $|1 - \eta a| < 1$ and positive.
- You see monotone exponential decay toward zero.
- This is classical, overdamped convergence.

✅ Exactly as expected.

---

#### 🟠 lr = 0.45 ≈ 0.9 / a (near the stability boundary)
- $(1-\eta a) \approx 0.1$, very small but still positive.
- The iterate almost collapses to zero in one step, then slowly creeps.
- This is critically damped / marginally stable behaviour.

✅ Also exactly right — fast contraction without oscillation.

---

#### 🟢 lr = 0.55 > 1 / a (oscillatory but stable)
- Now $(1-\eta a) < 0$ but still $|1-\eta a| < 1$.
- The sign flips on the first step → you see the overshoot into negative values.
- Subsequent oscillations decay rapidly toward zero.

✅ This is the textbook oscillatory convergence regime.

## 3. Stability of Gradient Descent as a Function of Learning Rate

This figure visualises how the learning rate controls the stability of gradient descent for a 1D quadratic loss.

Recall the update rule derived earlier:
$$x_{t+1} = (1 - \eta a)\,x_t,$$
where $a > 0$ is the curvature of the quadratic and $\eta$ is the learning rate.

The quantity $\rho$ where
$$\rho=|1 - \eta a|$$
is the **contraction factor** of the dynamical system: it tells us how much the iterate shrinks (or grows) at each step.

---

### What is being plotted?
- x-axis: the learning rate $\eta$
- y-axis: the contraction factor $|1 - \eta a|$
- Horizontal dashed line at $|1 - \eta a| = 1$: boundary between stability and divergence
- Vertical dashed line at $\eta = \tfrac{2}{a}$: the exact stability limit for gradient descent on a quadratic

---
### How to interpret the plot
Stable region:
$$|1 - \eta a| < 1 \quad \Longleftrightarrow \quad 0 < \eta < \tfrac{2}{a}$$
In this region, iterates converge to the minimum.
- Fastest contraction occurs near $\eta = \tfrac{1}{a}$, where the factor approaches zero and convergence is rapid.
- Oscillatory convergence happens when $1 - \eta a < 0$ but $|1 - \eta a| < 1$: the system flips sign each step but still shrinks.
- Divergence occurs once $\eta > \tfrac{2}{a}$: the contraction factor exceeds 1 and errors grow exponentially.

In [ ]:
a = 2.0
lrs = np.linspace(0.0, 2.0/a * 1.5, 200)
factor = np.abs(1 - lrs * a)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(lrs, factor, linewidth=2.5)
ax.axhline(1.0, linestyle="--", linewidth=2.0)
ax.axvline(2.0/a, linestyle="--", linewidth=2.0)
ax.set_title("Stability for 1D quadratic: $|1 - lr·a| < 1$", fontsize=14, fontweight="bold")
ax.set_xlabel("learning rate (lr)", fontsize=14, fontweight="bold")
ax.set_ylabel(r"contraction factor $|1-\eta a|$", fontsize=14, fontweight="bold")
ax.set_ylim(0, max(1.2, factor.max()))
style_ax(ax)
plt.show()

## 4. Gradient Descent in 2D: Geometry, Conditioning, and Trajectories

We now extend the 1D quadratic example to two dimensions, where gradient descent reveals richer geometric behaviour.

Consider the quadratic loss
$$L(x) = \tfrac{1}{2}\, x^\top A x,$$
with gradient
$$\nabla L(x) = A x.$$

Here, the matrix
$$A = \begin{pmatrix} 10 & 0 \\ 0 & 1 \end{pmatrix}$$
defines an ill-conditioned quadratic: the curvature along $x_1$ is ten times larger than along $x_2$.
This creates a narrow valley in parameter space.

---

### What the code does
- `quad2d(A)` defines a 2D quadratic loss and its gradient.
- We initialise gradient descent at $x_0 = (2, 2)$.
- We run gradient descent for several learning rates:
- a **small** step size (stable but slow),
- a **moderate** step size (faster convergence),
- a **large** step size near or beyond the stability limit.
- The resulting trajectories $(x_1^{(t)}, x_2^{(t)})$ are plotted directly in parameter space.

---
### How to read the trajectories

Each curve shows how gradient descent moves through $(x_1, x_2)$-space:
- For **small learning rates**, the trajectory follows the valley slowly, making many zig-zag steps.
- For **moderate learning rates**, convergence is faster but still constrained by the stiff direction.
- For **large learning rates**, the update becomes unstable in the high-curvature direction and may diverge.

The axes-aligned zig-zag pattern reflects the fact that:
- the gradient is much larger along $x_1$ than along $x_2$,
- yet the same learning rate is applied to both directions.

---
### Key insight: conditioning controls optimisation geometry

This example shows that:
- gradient descent stability is governed by the largest eigenvalue of $A$,
- while convergence speed in shallow directions is limited by the smallest eigenvalue.

As a result, a **single global learning rate** cannot be optimal for all directions when the problem is ill-conditioned.

This geometric mismatch is the root cause of:
- slow zig-zag convergence,
- sensitivity to learning rate choice,
- and the need for preconditioning and adaptive methods.

In [ ]:
def quad2d(A):
    A = torch.tensor(A)
    def fn(x):
        # x shape (2,)
        L = 0.5 * x @ (A @ x)
        g = A @ x
        return L, g
    return fn

A = np.array([[10.0, 0.0],
              [0.0, 1.0]])  # ill-conditioned: valley
f = quad2d(A)

x0 = torch.tensor([2.0, 2.0])
steps = 60
lrs = [0.05, 0.15, 0.25]  # last may diverge (since 2/lmax=0.2)

fig, ax = plt.subplots(figsize=(6,6))
for lr in lrs:
    xs, losses, gns = run_gd(x0, f, lr=lr, steps=steps)
    xy = xs.numpy()
    ax.plot(xy[:,0], xy[:,1], "-o", markersize=3, linewidth=2.0, label=f"lr={lr:.2f}")
ax.set_title("2D quadratic: trajectories in parameter space", fontsize=14, fontweight="bold")
ax.set_xlabel("x1", fontsize=14, fontweight="bold")
ax.set_ylabel("x2", fontsize=14, fontweight="bold")
ax.legend(prop={"size": 11, "weight": "bold"})
ax.axhline(0, linewidth=1.0)
ax.axvline(0, linewidth=1.0)
style_ax(ax)
plt.show()

### Behaviours of higher-dimension gradient descent: 

For higher-dimension gradient descent, it behaves like
$$x_{t+1}=(I-\eta A)x_t.$$

The convergence speed depends on the contraction factor 
$$\rho = \max_i |1-\eta\lambda_i|,$$
where $\lambda_i$ are eigenvalues of the matrix $A$.

In the process of gradient descent, neither too large or too small $\eta$ is optimal:
- Too large $\eta$ → **instability**
- Too small $\eta$ → **slow contraction**

There is an optimal range of learning rates that minimise $\rho$.

In fact, the **best** fixed step size is
$$\eta^*=\frac{2}{\lambda_{\text{max}}+\lambda_{\text{min}}}.$$

One can check the validity of this conclusion with the example below.

## 5. Visualising Gradient Descent on the Loss Landscape

To fully understand gradient descent as a dynamical system, it helps to see how the iterates move relative to the loss surface itself.

In this cell, we visualise gradient descent on a 2D quadratic by overlaying the optimisation trajectory on top of the level sets (contours) of the loss
$$L(x) = \tfrac{1}{2} x^\top A x.$$

---

### What the code does
- A grid over $(x_1, x_2)$-space is constructed.
- The quadratic loss is evaluated at each grid point to form contour lines.
- The gradient descent iterates $x^{(t)}$ are plotted on top of these contours.
- The same ill-conditioned matrix
$$A = \begin{pmatrix} 10 & 0 \\ 0 & 1 \end{pmatrix}$$
is used, producing elongated elliptical level sets.

---

### How to read the figure
- The contours represent points of equal loss.
- The path shows the actual trajectory taken by gradient descent.
- The steep curvature along $x_1$ forces large corrections in that direction.
- The shallow curvature along $x_2$ leads to slow progress across the valley.

The result is a characteristic zig-zag trajectory, where gradient descent repeatedly overshoots across the valley walls before making gradual progress toward the minimum.

---

### Key insight: gradient descent follows normals, not valleys

This visualisation highlights a fundamental limitation of vanilla gradient descent:
- The gradient always points **normal to the level sets**, not along their long axis.
- In ill-conditioned problems, this causes inefficient oscillations.
- Stability is controlled by the steepest direction, while speed is limited by the flattest one.

This geometric mismatch explains why:
- tuning the learning rate is difficult,
- convergence can be slow even for simple convex problems,
- and conditioning plays a central role in optimisation performance.

---

### Why this matters

This figure closes the conceptual loop:
- the update rule defines a dynamical system,
- the spectrum of A shapes the loss landscape,
- and the geometry of the contours determines the optimisation path.

Understanding this picture is essential before introducing:
- momentum,
- adaptive learning rates,
- or second-order methods.

In [ ]:
def contour_with_path(A, xs, xlim=(-2.5,2.5), ylim=(-2.5,2.5), levels=30):
    A = torch.tensor(A)
    x1 = np.linspace(*xlim, 200)
    x2 = np.linspace(*ylim, 200)
    X1, X2 = np.meshgrid(x1, x2)
    Z = np.zeros_like(X1)

    for i in range(X1.shape[0]):
        pts = torch.tensor(np.stack([X1[i], X2[i]], axis=-1))
        # f = 0.5 x^T A x
        Z[i] = (0.5 * (pts * (pts @ A.T)).sum(dim=1)).numpy()

    fig, ax = plt.subplots(figsize=(7,6))
    ax.contour(X1, X2, Z, levels=levels)
    path = xs.numpy()
    ax.plot(path[:,0], path[:,1], "-o", markersize=3, linewidth=2.5)
    ax.set_xlabel("x1", fontsize=14, fontweight="bold")
    ax.set_ylabel("x2", fontsize=14, fontweight="bold")
    ax.set_title("Contours + GD path", fontsize=14, fontweight="bold")
    style_ax(ax)
    plt.show()

A = np.array([[10.0, 0.0],[0.0, 1.0]])
f = quad2d(A)
x0 = torch.tensor([2.0, 2.0])
xs, losses, _ = run_gd(x0, f, lr=0.15, steps=40)
contour_with_path(A, xs)

## 6. The Gradient Vector Field: Local Dynamics Everywhere

So far, we have visualised one trajectory of gradient descent from a specific starting point.
In this cell, we step back and visualise the **entire dynamical system induced by gradient descent**.

We do this by plotting the vector field
$$-\nabla f(x),
\quad \text{where } f(x) = \tfrac{1}{2} x^\top A x.$$

This field assigns to every point $x$ the direction of steepest descent.

---
### What the code does
- A grid of points in $(x_1, x_2)$-space is constructed.
- At each point $x$, the gradient $\nabla f(x) = A x$ is computed.
- The arrows represent the negative gradient $-\nabla f(x)$, i.e. the instantaneous update direction of gradient descent.
- The same ill-conditioned quadratic matrix
$$A = \begin{pmatrix} 10 & 0 \\ 0 & 1 \end{pmatrix}$$
is used.

---
### How to read the figure
- Each arrow shows where gradient descent wants to move locally.
- Arrow length encodes gradient magnitude.
- The field is linear and points toward the origin, reflecting convexity.
- Directions are much steeper along $x_1$ than $x_2$, due to anisotropic curvature.

This means:
- updates are dominated by the stiff direction,
- step size must be chosen to satisfy the most restrictive curvature,
- slow directions are dragged along indirectly.

### Key insight: gradient descent is a flow

This plot reveals gradient descent as a discrete approximation to a continuous flow:
$$\dot{x}(t) = -\nabla f(x(t)).$$

The update rule
$$x_{t+1} = x_t - \eta \nabla f(x_t)$$
is simply a time-discretisation of this vector field.

Different learning rates correspond to different step sizes through the same field, not different objectives.

In [ ]:
def vector_field(A, xlim=(-2,2), ylim=(-2,2), n=15):
    A = torch.tensor(A)
    x1 = np.linspace(*xlim, n)
    x2 = np.linspace(*ylim, n)
    X1, X2 = np.meshgrid(x1, x2)
    U = np.zeros_like(X1)
    V = np.zeros_like(X2)

    for i in range(n):
        for j in range(n):
            x = torch.tensor([X1[i,j], X2[i,j]])
            g = A @ x
            d = (-g).numpy()
            U[i,j], V[i,j] = d[0], d[1]

    fig, ax = plt.subplots(figsize=(6,6))
    ax.quiver(X1, X2, U, V)
    ax.set_title("Vector field: -∇f(x)", fontsize=14, fontweight="bold")
    ax.set_xlabel("x1", fontsize=14, fontweight="bold")
    ax.set_ylabel("x2", fontsize=14, fontweight="bold")
    style_ax(ax)
    plt.show()

vector_field(np.array([[10.0, 0.0],[0.0, 1.0]]))

## 7. Non-convex 1D: a double-well and basins of attraction

This cell demonstrates a key difference between **convex** and **non-convex** optimisation:
the final point you converge to can depend strongly on the initialisation.

We define the classic double-well potential:
$$L(x) = \tfrac14 x^4 - \tfrac12 x^2$$
with gradient
$$\nabla L(x)=\frac{dL}{dx}=x^3 - x = x(x-1)(x+1).$$

---

### Geometry of the loss
- Critical points occur when $x^3-x=0$, i.e. $x\in\{-1,0,1\}$.
- $x=\pm 1$ are **local minima** (stable fixed points under gradient descent).
- $x=0$ is a **local maximum** (an unstable equilibrium).

So the loss has two valleys (“wells”) separated by a barrier at the origin.

---

### What this experiment is testing
We run gradient descent from multiple starting points $x_0$:
```python
x0s = [-1.5, -0.2, 0.2, 1.5]
```
using the same learning rate $\eta$, and track the trajectories $x_t$.

This illustrates that:
- initial points on the left tend to converge to the left minimum (-1),
- initial points on the right tend to converge to the right minimum (+1),
- points near 0 are “decided” by which side of the barrier they start on.

These are basins of attraction: regions of initial conditions that flow to the same stable fixed point.

---

### Why this matters for optimisation
In convex problems, “where you start” is mostly irrelevant — there is one global minimum.
In non-convex problems, optimisation is often about:
- which **basin** you land in,
- and how learning rate / noise / initialisation affect that outcome.

This is the first place where optimisation begins to look like **global behaviour emerging from local gradient steps** — a key theme for Part 2.

In [ ]:
def double_well():
    def fn(x):
        L = 0.25*x**4 - 0.5*x**2
        g = x**3 - x
        return L, g
    return fn

f = double_well()
x0s = [torch.tensor(-1.5), torch.tensor(-0.2), torch.tensor(0.2), torch.tensor(1.5)]
lr = 0.2

fig, ax = plt.subplots(figsize=(8,5))
for x0 in x0s:
    xs, losses, _ = run_gd(x0, f, lr=lr, steps=40)
    ax.plot(xs.squeeze().numpy(), "-o", markersize=3, linewidth=2.0, label=f"x0={float(x0):.1f}")
ax.set_title("Non-convex 1D: different initialisations → different basins", fontsize=14, fontweight="bold")
ax.set_xlabel("step t", fontsize=14, fontweight="bold")
ax.set_ylabel("x_t", fontsize=14, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

### Zooming in on the Unstable Critical Point

This cell probes the behaviour of gradient descent **in an infinitesimal neighbourhood of the critical point** $x = 0$.

We initialise gradient descent at:
- a tiny negative value ($-10^{-6}$),
- exactly at the critical point ($0$),
- and a tiny positive value ($+10^{-6}$),

and run the same update rule for many steps.

---

### What does this show?
- The trajectory starting **exactly at** $x=0$ remains fixed.
    - The gradient is zero, so gradient descent applies no update.
- Infinitesimal perturbations on either side **break symmetry**:
    - $-10^{-6}$ flows toward the left minimum,
    - $+10^{-6}$ flows toward the right minimum.
- The separation happens immediately, despite the perturbation being arbitrarily small.

This confirms that:
- $x=0$ is a stationary point,
- but an unstable equilibrium of the gradient descent dynamics.

---

### Why this matters

Gradient descent is not just minimising a function — it is a **discrete-time dynamical system**.
In non-convex landscapes:
- critical points can act as *separatrices* between basins,
- exact initialisation at such points is measure-zero,
- but nearby trajectories diverge decisively.

This sensitivity to initial conditions foreshadows:
- symmetry breaking,
- basin selection,
- and optimisation instability,

all of which will become central themes in Part 2.

## 8. Loss Decay and Stability in an Ill-Conditioned Quadratic

In [ ]:
f = double_well()

x0s = [torch.tensor(-1e-6),torch.tensor(0.0), torch.tensor(1e-6)]

lr = 0.2

fig, ax = plt.subplots(figsize=(8,5))
for x0 in x0s:
    xs, losses, _ = run_gd(x0, f, lr=lr, steps=steps)
    ax.plot(
        xs.squeeze().numpy(),
        "-o",
        markersize=3,
        linewidth=2.0,
        label=f"x0={x0}"
    )

ax.set_title(
    "Non-convex 1D: the unstable critical point at x = 0",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("step t", fontsize=14, fontweight="bold")
ax.set_ylabel(r"$x_t$", fontsize=14, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

This cell tracks how the objective value evolves over time for gradient descent applied to a 2D quadratic with highly unequal curvature:
$$f(x) = \tfrac{1}{2} x^\top A x, \qquad
A = \begin{pmatrix} 10 & 0 \\ 0 & 1 \end{pmatrix}.$$

We run gradient descent from the same initial point but with **different learning rates**, and plot the loss $f(x_t)$ on a logarithmic scale.

---
### What is being shown?
- Each curve corresponds to a different step size $\eta$.
- The vertical axis is logarithmic, so straight lines indicate exponential decay.
- The learning rates are chosen to span:
    - a conservative, clearly stable regime,
    - an aggressive but still stable regime,
    - a near-critical regime close to the stability boundary.

---
### How to interpret the results
- Smaller learning rates produce slow but smooth convergence.
- Larger learning rates accelerate early progress but approach instability.
- Near the stability limit, convergence becomes:
    - highly anisotropic,
    - dominated by the stiff (high-curvature) direction,
    - sensitive to step size choice.

Even though all runs converge, their **rates and transient behaviour differ dramatically**.

---
### Why this matters

This plot highlights a key optimisation principle:

> Stability is governed by the largest curvature,
> while convergence speed is governed by the smallest curvature.

As a result:
- ill-conditioned problems force a trade-off between speed and stability,
- loss curves alone can reveal conditioning and step-size limits,
- and learning rate tuning is fundamentally a **spectral problem**.

This tension will motivate:
- preconditioning,
- momentum,
- and adaptive methods in later tutorials.

In [ ]:
f = quad2d(np.array([[10.0, 0.0],[0.0, 1.0]]))
x0 = torch.tensor([2.0, 2.0])
lrs = [0.05, 0.15, 0.19]

fig, ax = plt.subplots(figsize=(8,5))
for lr in lrs:
    xs, losses, gns = run_gd(x0, f, lr=lr, steps=60)
    ax.plot(losses, linewidth=2.5, label=f"lr={lr:.2f}")
ax.set_yscale("log")
ax.set_title("Loss vs steps (log scale)", fontsize=14, fontweight="bold")
ax.set_xlabel("step t", fontsize=14, fontweight="bold")
ax.set_ylabel("loss", fontsize=14, fontweight="bold")
ax.legend(prop={"size": 11, "weight": "bold"})
style_ax(ax)
plt.show()

## 9. Empirical Stability of Gradient Descent vs Learning Rate

In this cell, we numerically test the **stability boundary of gradient descent** for a 2D quadratic objective and compare it against the theoretical prediction.

---

### What is being tested?

We consider the quadratic
$$f(x) = \tfrac{1}{2} x^\top A x,
\quad A = \begin{pmatrix} 10 & 0 \\ 0 & 1 \end{pmatrix},$$
whose largest eigenvalue is $\lambda_{\max} = 10.$

For gradient descent on a quadratic, theory predicts stability if and only if
$$\eta < \frac{2}{\lambda_{\max}} = 0.2.$$

---
### How the experiment works

For each learning rate $\eta$ in a grid:
1.	We run gradient descent for a fixed number of steps.
2.	We label the run as:
  - **stable (1)** if the trajectory remains finite,
  - **unstable (0)** if the loss or parameters explode.
3. We plot this binary outcome as a function of the learning rate.

This produces an **empirical stability diagram**.

---
### How to read the plot
- The flat region at `stable = 1` corresponds to learning rates where GD converges.
- The sharp transition to `stable = 0` marks the onset of instability.
- The printed vertical reference value
$$\eta = \tfrac{2}{\lambda_{\max}}$$
matches the observed boundary almost exactly.

---
### Why this matters

This experiment shows that:
- Gradient descent stability is **not heuristic** — it is sharply determined by curvature.
- The learning rate limit emerges as a **dynamical property**, not an implementation detail.
- Even in higher dimensions, GD behaves *exactly as predicted by eigenvalue analysis*.

This closes the loop between:
- analytical stability theory,
- numerical simulation,
- and the geometric interpretation of gradient descent as a dynamical system.

In the next sections, we will explore how this picture changes when:
- curvature varies across directions,
- objectives become non-quadratic,
- and local dynamics no longer capture global behaviour.

In [ ]:
def classify_lr(A, x0, lr, steps=80, explode_thresh=1e6):
    f = quad2d(A)
    xs, losses, _ = run_gd(x0, f, lr=lr, steps=steps)
    if np.any(~np.isfinite(losses)) or np.max(np.abs(xs.numpy())) > explode_thresh:
        return 0  # unstable
    return 1      # stable

A = np.array([[10.0, 0.0],[0.0, 1.0]])
x0 = torch.tensor([2.0, 2.0])
lrs = np.linspace(0.001, 0.35, 120)
stable = np.array([classify_lr(A, x0, lr) for lr in lrs])

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(lrs, stable, linewidth=2.5)
ax.set_title("Empirical stability vs learning rate (2D quadratic)", fontsize=14, fontweight="bold")
ax.set_xlabel("learning rate", fontsize=14, fontweight="bold")
ax.set_ylabel("stable? (1=yes, 0=no)", fontsize=14, fontweight="bold")
style_ax(ax)
plt.show()

print("Theoretical stability bound: lr < 2/lambda_max =", 2/np.linalg.eigvals(A).max())

## 🧭 Closing Remarks

In this tutorial, we reframed **gradient descent as a dynamical system**, rather than a training recipe.

By working through simple but revealing examples, we saw that:
- each gradient descent step is a deterministic update rule,
- convergence, oscillation, and divergence are governed by local curvature,
- learning rates control contraction factors, not just “speed,”
 and stability boundaries emerge sharply from eigenvalue structure.

Nothing in this analysis relied on neural networks, datasets, or stochasticity.

Yet the behaviours we observed — slow valleys, overshooting, instability, sensitivity to scale — already foreshadow the challenges of real optimisation problems.

The key takeaway is this:

> **Optimisation behaviour is already encoded in the geometry of the objective and the update rule.**

Once this perspective is internalised, many “mysteries” of training become predictable consequences of local dynamics.

In the next tutorials, we will build on this foundation by:
- extending the analysis to higher-dimensional and non-isotropic objectives,
- examining how conditioning and geometry shape convergence,
- and gradually moving from single-step dynamics to long-horizon optimisation behaviour.

What changes is complexity — not principle.